# **Integrated Gradients: Practice**

Welcome to the practice on the integrated gradients method! Counting the explanation methods for deep models, this is our fifth practice :)
Congratulations, you have come a long way!

So, the whole notebook is devoted to the **Integrated Gradients** method — the first axioms-based method in our course. As a reminder, there are 5 axioms in total, and the formula for computing Integrated Gradients is:
![ig_image](https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/assets/9e86fa78-8c24-4b69-8743-1e9484caa629.png)

In this lesson you will:
- Implement the Integrated Gradients method from scratch as your own function;
- Practise applying the method from the CAPTUM library — a library devoted to Interpretability for PyTorch models;
- Feel the difference between using different baselines for the method;

It will be interesting!
 Happy coding!

First of all, let us install the captum library. We will need it at the end of the lesson.

In [ ]:
!pip install captum -q

And let us import everything we need for the work.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import requests
from datetime import datetime
from tqdm import tqdm
from io import BytesIO
import urllib
from PIL import Image
%matplotlib inline

import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF

from torchvision import models

from captum.attr import IntegratedGradients
from captum.attr import visualization as viz

from torchvision.models import resnet50, swin_t

As in the first practices, we will work with pretrained networks. However, so as not to create the illusion that only convolutional neural networks can be explained, this time we will also work with a transformer-type model — `SWIN`. You can find an overview of the architecture [here](https://habr.com/ru/articles/599057/).

In [ ]:
resnet = resnet50(weights='IMAGENET1K_V1') # as before, we use ResNet
swin_net = swin_t(weights='IMAGENET1K_V1') # and let us load Swin

# Inference mode is a must: in train mode BatchNorm uses the statistics of a batch
# of one image, and the prediction stops matching the real one.
resnet.eval()
swin_net.eval();

We will work with two images. Our first example will be a piggy, and the second one a cat.

In [ ]:
hog_url = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/hog.jpg'

hog_image_bytes = requests.get(hog_url).content
hog_image = Image.open(BytesIO(hog_image_bytes))

plt.axis('off')
plt.imshow(hog_image);

1. Load and visualise the cat yourself, using the link: https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/cat.jpg

In [ ]:
cat_url = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/cat.jpg'

cat_image_bytes = # Your code here
cat_image = # Your code here

plt.axis('off')
plt.imshow(cat_image);

We will also additionally need the classes and indices from Imagenet in order to verify the predictions of the networks.

In [ ]:
url = "https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/imagenet_classes.txt"
urllib.request.urlretrieve(url, "imagenet_classes.txt")

with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]

Let us go through the standard steps — preprocess the images before feeding them to the models with the help of the transform function.

In [ ]:
#Means
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Let us preprocess the image
transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)), # Resizing the image
    torchvision.transforms.ToTensor(),          # Conversion to a tensor
    torchvision.transforms.Normalize(mean, std) # Normalisation
])

In [ ]:
hog_input = transform(hog_image) # let us preprocess the images
cat_input = transform(cat_image)

fig, ax = plt.subplots(1, 2, figsize=(12, 8))

ax[0].imshow(hog_input.permute((1, 2, 0))) ; # let us look at what we got after normalisation for the piggy
ax[0].set_title('Hog after transform')

ax[1].imshow(cat_input.permute((1, 2, 0))) # and for the cat
ax[1].set_title('Cat after transform');

To finish with, for both images let us add the batch dimension and explicitly tell pyTorch to compute and store the gradients for them.

In [ ]:
hog_input.unsqueeze_(0);  # let us add the batch dimension
cat_input.unsqueeze_(0);

hog_input.requires_grad = True  # we explicitly tell PyTorch to compute and store the gradients for the input image
cat_input.requires_grad = True

2. Get the prediction for the models. Which indices did resnet and swin predict? Enter the answers in the trainer for questions 1, 2, 3.

We will treat the classes hog (341) and tiger cat (282) as the target ones.

In [ ]:
cat_resnet_index = # Your code here
cat_swin_index = # Your code here

print(f'ResNet prediction for cat {cat_resnet_index} (index), {categories[cat_resnet_index]} (class)')
print(f'SwinNet prediction for cat {cat_swin_index} (index), {categories[cat_swin_index]} (class)')

In [ ]:
hog_resnet_index = # Your code here
hog_swin_index = # Your code here

print(f'ResNet prediction for hog {hog_resnet_index} (index), {categories[hog_resnet_index]} (class)')
print(f'SwinNet prediction for hog {hog_swin_index} (index), {categories[hog_swin_index]} (class)')

As we can see, the SWIN model approached the solution of the task at hand in a very unexpected way! Let us look at which classes made it into the top-3 for the cat and at what exactly confused SWIN.

3. Get the top-3 predictions of the Swin Transformer for the cat class. By how much is the first index larger than the second? Enter the value you get in step 5, to 4 decimal places.

In [ ]:
top3_swin_cat = # Your code here

print('Top3 Swin for cat image: ', top3_swin_cat[:])

Decode the prediction of the swin transformer for the 2nd and the 3rd values.

Whether to consider it an error that the target class did not make it into the top-3 for the swin model depends on the task. For example, if we were recognising exactly the breeds of cats, then the model clearly still has to be fine-tuned, or something else has to be thought up.

Let us look at what made the model include inanimate objects in the top-3, using Integrated Gradients.

**1. Integrated gradients: a custom implementation.**

Before writing the method itself, let us prepare the objects that we will treat as *baseline* ones. As we noted in the theory, the outputs of the method (that is, our explanations) will be sensitive to different baselines. The time has come to check this in practice!

**Among the candidate objects for the baseline we will consider 4:**
-  A completely zero image (zero baseline)
- An image based on Gaussian noise (noise baseline)
- An image based on the channel means (mean baseline)
- An image based on a random distribution (random baseline)

Others can be considered too. The general recommendation is this — the zero baseline often works well, but sometimes using the others makes it possible to refine some regions. It is better to consider several, including the zero baseline.

In [ ]:
def gaussian_noise(x, var):
  """
  Gaussian noise based on the image x
  """

  return torch.normal(0, var, size=x.shape)


def random_baseline(x, low, high):
    """
    A random distribution based on the image x
    """

    return np.random.uniform(low, high,x.shape)

Let us implement and visualise the 4 baseline objects for the image with the cat one after another, since we will only investigate the predictions on this object.

In [ ]:
cat_zero_baseline = cat_input * 0 #Zero baseline


cat_noise_baseline = torch.ones_like(cat_input) # Noise baseline
cat_noise_baseline += gaussian_noise(cat_noise_baseline, 0.1)

cat_mean_baseline = torch.ones_like(cat_input) # Mean baseline
cat_mean_baseline *= torch.mean(cat_input, axis=1)

cat_random_baseline = random_baseline(x=cat_input, low=0, high=15) # Random baseline
cat_random_baseline = torch.from_numpy(cat_random_baseline).float()


And let us write a simple function for visualising tensors, in order to automate this process.

In [ ]:
def vizualize_tensor(tensor):

  if len(tensor.shape) == 4:
    return tensor.squeeze(0).permute((1, 2, 0)).detach().numpy()
  else:
    return tensor.permute((1, 2, 0)).detach().numpy()

In [ ]:
# Creating the figure and a 1x4 grid
fig, axes = plt.subplots(1, 4, figsize=(16, 8))

# Displaying each image, row 1
axes[0].imshow(vizualize_tensor(cat_zero_baseline))
axes[0].set_title('Zero baseline')

axes[1].imshow(vizualize_tensor(cat_noise_baseline))
axes[1].set_title('Noise baseline')

axes[2].imshow(vizualize_tensor(cat_mean_baseline))
axes[2].set_title('Mean baseline')

axes[3].imshow(vizualize_tensor(cat_random_baseline))
axes[3].set_title('Random baseline');

Now the time has come to implement Integrated gradients. The formula we need is:
![ig_image](https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/assets/9e86fa78-8c24-4b69-8743-1e9484caa629.png)

What we need is:
1. To get the objects x and the baselines for them
2. To compute the differential in the form of a weighted sum

Implement the computation of the sum. As the argument X you have to add the value **inside** the function, that is, $(x' + \frac{k}{m}(x-x'))$

In [ ]:
from tqdm import tqdm


def compute_integrated_gradient(batch_x, batch_blank, model, index_interested, m=100):

    mean_grad = 0

    baseline_diff = (batch_x - batch_blank)/m # the first two factors

    for i in tqdm(range(1, m + 1)):

        x =  """ Your code here """ # the value that will be fed to the function in the numerator
        x.requires_grad_ = True                           # we explicitly tell pyTorch to compute the gradients
        y = model(x)                                      # we get the model prediction

        score = y[0, index_interested]                    # we compute the gradient with respect to the feature we are interested in
        (grad,) = torch.autograd.grad(score, x)
        mean_grad += grad                                 # we add to the variable a term that is ALREADY divided by the number of approximation steps

    mean_grad /= m


    integrated_gradients = (batch_x - batch_blank) * mean_grad # we put all the factors together

    return integrated_gradients

This task tests the correctness of the implementation of the line above. As the answer in the trainer, give the value printed by print for `m = 10`.

In [ ]:
def test_compute_x():
    batch_x = torch.ones((1, 3, 224, 224)) * 255
    batch_blank = torch.zeros((1, 3, 224, 224))
    m = 10

    #
    expected_x = batch_blank + 5 / 10 * (batch_x - batch_blank)

    #
    computed_x = None
    for j in range(1, 5 + 1):
        computed_x = batch_blank + j / m * (batch_x - batch_blank)

    assert torch.allclose(computed_x, expected_x), "Error in the computation of the variable x"
    print(computed_x[0, 0, 0, 0])

test_compute_x()

In [ ]:
categories[532], categories[285], categories[495]

So, let us look at what made the model include in the top-3 two inanimate objects with the indices 532 and 495, using the function we have built.

In [ ]:
cat_ig_zero532 = compute_integrated_gradient(cat_input, cat_zero_baseline, swin_net, 532, 50)
cat_ig_noise532  = compute_integrated_gradient(cat_input, cat_noise_baseline, swin_net, 532, 50)
cat_ig_mean532 = compute_integrated_gradient(cat_input, cat_mean_baseline, swin_net, 532, 50)
cat_ig_random532  = compute_integrated_gradient(cat_input, cat_random_baseline.float(), swin_net, 532, 50)

Let us look at the maps we got for the index 532.

In [ ]:
# CAT + SWIN NET

# Creating the figure and a 2x4 grid
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

# Displaying each image, row 1
# Displaying each image, row 1
axes[0, 0].imshow(vizualize_tensor(cat_zero_baseline))
axes[0, 0].set_title('Zero baseline')

axes[0, 1].imshow(vizualize_tensor(cat_noise_baseline))
axes[0, 1].set_title('Noise baseline')

axes[0, 2].imshow(vizualize_tensor(cat_mean_baseline))
axes[0, 2].set_title('Mean baseline')

axes[0, 3].imshow(vizualize_tensor(cat_random_baseline))
axes[0, 3].set_title('Random baseline');


# Displaying each image, row 2
axes[1, 0].imshow(vizualize_tensor(cat_ig_zero532)*10)
axes[1, 0].set_title('Zero ig')

axes[1, 1].imshow(vizualize_tensor(cat_ig_noise532)*10)
axes[1, 1].set_title('Noise ig')

axes[1, 2].imshow(vizualize_tensor(cat_ig_mean532)*10)
axes[1, 2].set_title('Mean ig')

axes[1, 3].imshow(vizualize_tensor(cat_ig_random532)*10)
axes[1, 3].set_title('Random ig')


# Turning off the axes for all the images
for ax in axes.flat:
    ax.axis('off')

# Displaying the images
plt.tight_layout()
plt.show();

And let us look at the maps we got for the index 495.

In [ ]:
swin_net.zero_grad()

cat_ig_zero495 = compute_integrated_gradient(cat_input, cat_zero_baseline, swin_net, 495, 50)
cat_ig_noise495 = compute_integrated_gradient(cat_input, cat_noise_baseline, swin_net, 495, 50)
cat_ig_mean495  = compute_integrated_gradient(cat_input, cat_mean_baseline, swin_net, 495, 50)
cat_ig_random495 = compute_integrated_gradient(cat_input, cat_random_baseline.float(), swin_net, 495, 50)

In [ ]:
# CAT + SWIN NET

# Creating the figure and a 2x4 grid
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

# Displaying each image, row 1
# Displaying each image, row 1
axes[0, 0].imshow(vizualize_tensor(cat_zero_baseline))
axes[0, 0].set_title('Zero baseline')

axes[0, 1].imshow(vizualize_tensor(cat_noise_baseline))
axes[0, 1].set_title('Noise baseline')

axes[0, 2].imshow(vizualize_tensor(cat_mean_baseline))
axes[0, 2].set_title('Mean baseline')

axes[0, 3].imshow(vizualize_tensor(cat_random_baseline))
axes[0, 3].set_title('Random baseline');


# Displaying each image, row 2
axes[1, 0].imshow(vizualize_tensor(cat_ig_zero495)*10)
axes[1, 0].set_title('Zero ig')

axes[1, 1].imshow(vizualize_tensor(cat_ig_noise495)*10)
axes[1, 1].set_title('Noise ig')

axes[1, 2].imshow(vizualize_tensor(cat_ig_mean495)*10)
axes[1, 2].set_title('Mean ig')

axes[1, 3].imshow(vizualize_tensor(cat_ig_random495)*10)
axes[1, 3].set_title('Random ig')


# Turning off the axes for all the images
for ax in axes.flat:
    ax.axis('off')

# Displaying the images
plt.tight_layout()
plt.show();

We can see that the model pays attention to the *setting of the room* rather than to the central object. In a real task, one of the possible solutions could be validating the input images by a rule such as (for example) "if the object occupies a small place in the picture, then crop the image".

Let us see whether cropping the image really works!

Choose the necessary `crop_value`

In [ ]:
width, height = cat_image.size

crop_value = # Your code here

left = (width - crop_value)/2
top = (height - crop_value)/2
right = (width + crop_value)/2
bottom = (height + crop_value)/2

cropped_cat = cat_image.crop((left, top, right, bottom))

cropped_cat_input = transform(cropped_cat)

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(vizualize_tensor(cropped_cat_input))

In [ ]:
cropped_cat_input.unsqueeze_(0);  # let us add the batch dimension

cropped_cat_input.requires_grad = True  # we explicitly tell PyTorch to compute and store the gradients for the input image

In [ ]:
print('Model prediction for the cropped image:', categories[int(torch.argmax(swin_net(cropped_cat_input)))])

In [ ]:
torch.topk(swin_net(cropped_cat_input), axis=1, k=3)

The right cropping will not only help the model to recognise the cat, but will also throw all the inanimate objects out of the top3!

**2. Integrated gradients: working with the captum library.**

Finally, let us practise using the library.

To work with the Integrated gradients method, you will need IntegratedGradients itself (we already imported it in the first cell) and the .attribute method.



```
attribute(inputs, baselines=None, target=None, additional_forward_args=None,
n_steps=50, method='gausslegendre', internal_batch_size=None, return_convergence_delta=False)

```
where:
- inputs — the input images
- baselines - the baselines of the images
- target — the class for which the explanation will be built
- additional_forward_args — in case the model returns not only the standard output, but other Python objects as well
- n_steps - the number of approximation steps
- method — the method of computing the integral
- internal_batch_size — for splitting inputs, if required
- return_convergence_delta — the difference between the total approximate and the true integrated gradients

Let us implement the computation and measure the time.

In [ ]:
start_time = datetime.now()

integrated_gradients = IntegratedGradients(swin_net)
swin_net.zero_grad()
captum_ig_attributions = integrated_gradients.attribute(inputs=cat_input,
                                                        baselines=cat_zero_baseline,
                                                        target=532,
                                                        n_steps=50,
                                                        method='riemann_right')

print(f'Duration: {datetime.now() - start_time}')


And, of course, let us visualise the result!

In [ ]:
plt.imshow(vizualize_tensor(captum_ig_attributions)*10)

# **Conclusion**

So, in this lesson you have:
- Implemented the Integrated Gradients method from scratch
- Worked with different baselines for the Integrated Gradients method
- Solved the problem of the incorrect class determined by the Swin Transformer model
- Got acquainted with the captum library.

It is also important to note that in our example and our practice, for the image with the cat the important zones were shown best of all with the zero baseline. **But that does not mean that the zero baseline always leads to success!** The baseline has to be chosen for each specific image on your own. Besides, you have probably noticed that by default we multiplied the maps by 10 without explaining this action. This is due to the fact that the values obtained can be **very small**, and instead of the important zones you will just see a black square. Another way of dealing with this is to use the capabilities of captum, which we will discuss in the next practice.

*(an illustration for the analysis — the result is reproduced by the cells below)*

# **Part 3. The choice of the baseline is not an implementation detail**

The theory says: **a feature that coincides with the baseline gets exactly zero attribution**,
no matter how important it may be. The reason is visible in the formula — the attribution is the product of the mean
gradient and the difference $(x_i - x'_i)$. Let us check this with a number.

In [ ]:
# Let us take the zero baseline and deliberately zero out a 60x60 square in the image.
# Inside it the input WILL COINCIDE with the baseline — which means the attribution there must be zero.
patched = cat_input.clone()
patched[:, :, 80:140, 80:140] = 0.0

ig_patched = compute_integrated_gradient(patched, cat_zero_baseline, swin_net, 532, 50)

inside = ig_patched[:, :, 80:140, 80:140]
print('sum of the absolute attributions INSIDE the region that coincided with the baseline:',
      float(  # Your code here
      ))

**Quiz.** What is the sum of the absolute attributions inside the region that coincided with the baseline?
The answer goes into the trainer.

This is exactly the blind spot. With a black baseline, such a region turns out to be everything dark:
the pupil, the shadow, the dark fur. The method will silently declare them unimportant.

In [ ]:
# How much attribution goes to the dark pixels with different baselines
dark = (cat_input.mean(1, keepdim=True) < cat_input.mean()).float()

for name, base in [('zero', cat_zero_baseline), ('noise', cat_noise_baseline),
                   ('mean', cat_mean_baseline), ('random', cat_random_baseline.float())]:
    ig = compute_integrated_gradient(cat_input, base, swin_net, 532, 50)
    share = float((ig.abs() * dark).sum() / (ig.abs().sum() + 1e-12))
    print(f'{name:7s}  share of the attribution in the dark regions: {share:.3f}')

# **Part 4. Checking completeness: convergence delta**

The sum of the attributions must be equal to $F(x) - F(x')$. The integral is computed as a Riemann sum,
so the equality is approximate — and the residual has to be checked rather than assumed.

In [ ]:
@torch.no_grad()
def score(model, x, idx):
    return float(model(x)[0, idx])

target = 532
delta_target = score(swin_net, cat_input, target) - score(swin_net, cat_zero_baseline, target)

for m in (10, 50, 200):
    ig = compute_integrated_gradient(cat_input, cat_zero_baseline, swin_net, target, m)
    got = float(ig.sum())
    print(f'm = {m:>3}   sum of attributions {got:8.4f}   F(x) - F(x\') {delta_target:8.4f}   '
          f'residual {abs(got - delta_target):.4f}')

Look at the numbers carefully: at $m=50$ the residual is almost zero, while at $m=200$ it grows again.

A Riemann sum does not promise monotone convergence: the gradient of the network along the path is not smooth, and a finer
partition simply lands on other points of this unevenness. The practical conclusion is the opposite of
"let us just take more steps": $m$ is chosen **by the residual**, and not increased blindly.

**Quiz.** What did this run show? The answer goes into the trainer.